In [1]:
# ############## Inicialização ########################

%matplotlib qt5

import numpy as np
import matplotlib.pyplot as mplt
import mplcursors

from siriuspy.sofb.csdev import SOFBFactory
from siriuspy.fofb.csdev import HLFOFBConst
from siriuspy.clientconfigdb import ConfigDBClient
from siriuspy.clientarch import PVDataSet, Time

# Cria objeto para pegar informações sobre BPMs e Corretoras
sofb = SOFBFactory.create('SI')
fofb = HLFOFBConst()

# Cria cliente para pegar a matriz resposta do SOFB
clt = ConfigDBClient(config_type='si_orbcorr_respm')
sofb_orm = np.array(clt.get_config_value('ref_respmat'))
fofb_orm = np.array(clt.get_config_value(
    'ref_respmat', config_type='si_fastorbcorr_respm'
))

# Cria cliente para pegar dados do archiver
posx = [b + ':PosX-Mon' for b in sofb.bpm_names] 
posy = [b + ':PosY-Mon' for b in sofb.bpm_names] 
schk = [b + ':KickRef-Mon' for b in sofb.ch_names] 
scvk = [b + ':KickRef-Mon' for b in sofb.cv_names] 
fchk = [b + ':KickRef-Mon' for b in fofb.ch_names] 
fcvk = [b + ':KickRef-Mon' for b in fofb.cv_names]
pvs = posx + posy + schk + scvk + fchk + fcvk
pvd = PVDataSet(pvs)

# ############# Escolha de parâmetros ##################
include_fofb = False
average_interval = 2*60  # [s]
tim0 = Time(2026, 10, 9, 4, 10, 0, 0)
tim1 = Time(2026, 10, 9, 15, 30, 0, 0)

# ############## Pega e processa os dados ###############

# Pega os dados no tempo passado
pvd.time_start = tim0
pvd.time_stop = tim0 + average_interval
pvd.update()
ox0 = np.array([pvd[n].value.mean() for n in posx]) / 1e3  # Orbita X
oy0 = np.array([pvd[n].value.mean() for n in posy]) / 1e3  # Orbita Y
sh0 = np.array([pvd[n].value.mean() for n in schk])  # Forças SOFB CH
sv0 = np.array([pvd[n].value.mean() for n in scvk])  # Forças SOFB CV
fh0 = np.array([pvd[n].value.mean() for n in fchk])  # Forças FOFB CH
fv0 = np.array([pvd[n].value.mean() for n in fcvk])  # Forças FOFB CV

# Pega os dados no tempo atual
pvd.time_start = tim1
pvd.time_stop = tim1 + average_interval
pvd.update()
ox1 = np.array([pvd[n].value.mean() for n in posx]) / 1e3  # Orbita X
oy1 = np.array([pvd[n].value.mean() for n in posy]) / 1e3  # Orbita Y
sh1 = np.array([pvd[n].value.mean() for n in schk])  # Forças SOFB CH
sv1 = np.array([pvd[n].value.mean() for n in scvk])  # Forças SOFB CV
fh1 = np.array([pvd[n].value.mean() for n in fchk])  # Forças FOFB CH
fv1 = np.array([pvd[n].value.mean() for n in fcvk])  # Forças FOFB CV

# Calcula a variação de órbita e de forças
dsh = sh1 - sh0
dsv = sv1 - sv0
dfh = fh1 - fh0
dfv = fv1 - fv0
dox = ox1 - ox0
doy = oy1 - oy0
dsc = np.r_[dsh, dsv]
dfc = np.r_[dfh, dfv]
dor = np.r_[dox, doy]

# Calcula a órbita reconstruída e a total
dors = sofb_orm[:, :-1] @ dsc
dorf = fofb_orm[:, :-1] @ dfc
dor_tot = dor + dors + dorf * include_fofb

# ################## Faz gráficos ##########################

title = f"from {tim0.strftime('%Y/%m/%d, %H:%M:%S')} "
title += f"to {tim1.strftime('%Y/%m/%d, %H:%M:%S')} "
title += f'(Avg. over {average_interval:.1f} sec.)'

# Gráfico de órbita
fig, (ax, ay) = mplt.subplots(2, 1, sharex=True, figsize=(10, 6))

# Plota plano horizontal
linhas = []
linhas += ax.plot(sofb.bpm_pos, dor[:160], '-o', label='Read by BPMs')
linhas += ax.plot(sofb.bpm_pos, dor_tot[:160], '-o', label='Total')
ax.set_ylabel("Horizontal [um]")
ax.grid(True, ls='--', alpha=0.4, color='k')

# Plota plano vertical:
linhas += ay.plot(sofb.bpm_pos, dor[160:], '-o', label='Read by BPMs')
linhas += ay.plot(sofb.bpm_pos, dor_tot[160:], '-o', label='Total')
ay.set_ylabel("Vertical [um]")
ay.grid(True, ls='--', alpha=0.4, color='k')
ay.set_xlabel('BPM Position [m]')

# Legenda e Título
ay.legend(loc='best')
ax.set_title('Orbit Variation ' + title)
fig.tight_layout()
fig.show()

# Gráfico de força de corretoras lentas
fig, (ax, ay) = mplt.subplots(2, 1, sharex=True, figsize=(10, 6))

# Plota plano horizontal
lsh = ax.plot(sofb.ch_pos, dsh, '-o')[0]
ax.set_ylabel("Horizontal [urad]")
ax.grid(True, ls='--', alpha=0.4, color='k')

# Plota plano vertical:
lsv = ay.plot(sofb.cv_pos, dsv, '-o')[0]
ay.set_ylabel("Vertical [urad]")
ay.grid(True, ls='--', alpha=0.4, color='k')
ay.set_xlabel('Correctors Position [m]')

# Título
ax.set_title('Slow Correctors Variation ' + title)
fig.tight_layout()
fig.show()

if include_fofb:
    # Gráfico de força de corretoras rápidas
    fig, (ax, ay) = mplt.subplots(2, 1, sharex=True, figsize=(10, 6))

    # Plota plano horizontal
    lfh = ax.plot(fofb.ch_pos, dfh, '-o')[0]
    ax.set_ylabel("Horizontal [urad]")
    ax.grid(True, ls='--', alpha=0.4, color='k')

    # Plota plano vertical:
    lfv = ay.plot(fofb.cv_pos, dfv, '-o')[0]
    ay.set_ylabel("Vertical [urad]")
    ay.grid(True, ls='--', alpha=0.4, color='k')
    ay.set_xlabel('Correctors Position [m]')

    # Título
    ax.set_title('Fast Correctors Variation ' + title)
    fig.tight_layout()
    fig.show()

# Adiciona Tooltips
linhas += [lsh, lsv]
nomes = {lsh: sofb.ch_nicknames, lsv: sofb.cv_nicknames}
if include_fofb:
    linhas += [lfh, lfv]
    nomes |= {lfh: fofb.ch_nicknames, lfv: fofb.cv_nicknames}
cursor = mplcursors.cursor(linhas, hover=mplcursors.HoverMode.Transient)

@cursor.connect("add")
def on_add(sel):
    line = sel.artist
    idx = int(sel.index + 0.5)
    nome = nomes.get(line, sofb.bpm_nicknames)[idx]
    sel.annotation.set_text(nome)
 